In [4]:
import json
from notion2pandas import Notion2PandasClient

In [6]:
def get_text(notion_blocks):
    if not notion_blocks or 'results' not in notion_blocks:
        return ''

    for block in notion_blocks['results']:
        block_type = block.get('type')
        block_data = block.get(block_type, {})
        rich_texts = block_data.get('rich_text', [])
        if rich_texts:
            # Prende il contenuto testuale del primo rich_text
            return ''.join(rt.get('plain_text', '') for rt in rich_texts)

    return ''

In [19]:
with open('notion_data.json', 'r') as notionFile:
    notion_data = json.load(notionFile)
    token = notion_data.get('notion_api_token')
    database_id = notion_data.get('rgj25').get('id_database_pages_db')
custom_block_prop = {
    'inside_text': get_text
}

sort_by_name = {
    "sorts": [
        {
            "property": "Name",
            "direction": "ascending"  # oppure "descending"
        }
    ]
}

n2p = Notion2PandasClient(auth=token)


    # Creazione DataFrame
df = n2p.from_notion_DB_to_dataframe_kwargs(database_id, filter_params= sort_by_name, columns_from_blocks=custom_block_prop)
    
    

In [20]:
df

,Content pages DB,Name,PageID,inside_text,Row_Hash
0,['28944e81-37da-8076-a5d8-fee4ce1b2ad6'],00-Phase,28944e81-37da-8044-824d-d9b3d55dab07,dio maledetto,190468824
1,['28944e81-37da-80b4-a874-ca518e3bc7ef'],01-Phase,28944e81-37da-809a-a7a1-ee2040e34802,🤖✏️❤️😏👈🏻😘😘🙊😒,8134548875
2,['28944e81-37da-80cd-b8ed-cee722a54954'],02-Phase,28944e81-37da-8071-a826-d2bf17b26b74,,3427959058


In [ ]:
def write_notion_page(phaseName, notion_blocks) -> tuple[str, bool]:

In [40]:
copy_page_content_full(n2p, '28944e81-37da-8076-a5d8-fee4ce1b2ad6', '28944e81-37da-8044-824d-d9b3d55dab07')


KeyboardInterrupt



In [41]:
import time
import requests
from notion_client import Client

def copy_page_content_full(notion: Client, source_page_id: str, target_page_id: str):
    """
    Copies all blocks from one Notion page to another.
    Notion-hosted files/images are downloaded and re-uploaded.
    External files/images are copied as links.
    """

    def _wait_for_upload_completion(file_upload_id: str, poll_interval=5, max_wait_time=300):
        """Wait for a single_part file upload to complete."""
        start_time = time.monotonic()
        while time.monotonic() - start_time < max_wait_time:
            status_resp = notion.file_uploads.retrieve(file_upload_id=file_upload_id)
            status = status_resp.get("status")
            if status == "uploaded":
                return
            elif status == "failed":
                raise Exception(f"Upload failed: {status_resp}")
            time.sleep(poll_interval)
        raise TimeoutError("File upload timed out")

    def clean_media_block(block_type: str, block_content: dict):
        """Prepare a media block (image, file, pdf, video, audio) for appending."""
        file_info = block_content.get("file")
        external_info = block_content.get("external")

        if block_type != "image":
            return None  # Puoi estendere per altri media se vuoi

        if file_info and "url" in file_info:
            # Notion-hosted → scarica e ricarica
            filename = "image.jpg"
            response = requests.get(file_info["url"])
            response.raise_for_status()
            content = response.content
            upload_resp = notion.file_uploads.create(
                mode="single_part",
                filename=filename,
                content=content
            )
            file_upload_id = upload_resp["id"]
            _wait_for_upload_completion(file_upload_id)
            return {
                "object": "block",
                "type": "image",
                "image": {"type": "file_upload", "file_upload": {"id": file_upload_id}}
            }

        elif external_info and "url" in external_info:
            # External → copia direttamente
            return {
                "object": "block",
                "type": "image",
                "image": {"type": "external", "external": {"url": external_info["url"]}}
            }

        return None

    def copy_block_recursive(src_block_id: str, dst_parent_id: str):
        """Recursively copy all child blocks."""
        children = notion.blocks.children.list(block_id=src_block_id)["results"]

        for child in children:
            block_type = child["type"]
            block_content = child.get(block_type, {})

            if block_type == "image":
                media_block = clean_media_block(block_type, block_content)
                if not media_block:
                    continue
                new_block = media_block
            else:
                # Copy normal block content
                new_block = {"object": "block", "type": block_type}
                new_block[block_type] = {}
                for k, v in block_content.items():
                    if k not in (
                        "id",
                        "created_time",
                        "last_edited_time",
                        "created_by",
                        "last_edited_by",
                        "has_children",
                    ):
                        new_block[block_type][k] = v

            # Append the block
            created = notion.blocks.children.append(
                block_id=dst_parent_id,
                children=[new_block]
            )
            new_block_id = created["results"][0]["id"]

            # Recursively copy children if present
            if child.get("has_children"):
                copy_block_recursive(child["id"], new_block_id)

    # Start recursive copy
    copy_block_recursive(source_page_id, target_page_id)
    print("✅ Page content copied successfully (with media).")


In [16]:
def copy_page_with_files(notion: Notion2PandasClient, source_page_id: str, target_page_id: str):
    def copy_block_recursive(src_block_id, dst_parent_id):
        children = notion.blocks.children.list(block_id=src_block_id)["results"]
        for child in children:
            block_type = child["type"]
            block_content = child.get(block_type, {})
            
            new_block = {
                "object": "block",
                "type": block_type,
            }
            
            # se è un blocco file / image / media: gestiamo upload
            if block_type in ("file", "image", "pdf", "video", "audio"):
                # caso 1: file già caricato in Notion (type "file")
                file_obj = block_content.get("file") or block_content.get("external")
                if file_obj:
                    # se file interno (Notion-hosted) o esterno, puoi copiarlo direttamente
                    new_block[block_type] = {
                        **block_content,
                        "file": file_obj,  # mantiene collegamento al file
                        "external": block_content.get("external")
                    }
                else:
                    # caso 2: devi fare upload nuovo (presumendo che tu abbia il file sorgente)
                    # dovresti avere accesso al file fisico o URL per far upload...
                    # Esempio:
                    upload_resp = notion.file_uploads.create(file=...)  # ipotetico
                    file_upload_id = upload_resp["id"]
                    new_block[block_type] = {
                        **block_content,
                        "type": "file_upload",
                        "file_upload": {"id": file_upload_id}
                    }
            else:
                # blocco “normale” (paragrafo, heading, ecc.)
                new_block[block_type] = block_content.copy()
            
            # Rimuovi proprietà non valide per la creazione
            for field in ("id", "created_time", "last_edited_time", "has_children"):
                new_block[block_type].pop(field, None)
            
            # Appendi al blocco/parent target
            created = notion.blocks.children.append(
                parent_id=dst_parent_id,
                children=[new_block]
            )
            new_block_id = created["results"][0]["id"]
            
            # Ricorsivamente copia figli se presenti
            if child.get("has_children"):
                copy_block_recursive(child["id"], new_block_id)
    
    # Avvio copia
    copy_block_recursive(source_page_id, target_page_id)

NameError: name 'Client' is not defined